# Create and save dictionaries of author names

## Load libraries and data

In [14]:
# import libraries
import pickle
import statistics
import matplotlib.pyplot as plt
import re
import pandas as pd
from collections import Counter
import numpy as np
from collections import OrderedDict
import math

In [15]:
# read in datasets
aus_in = '../data/aus/clean/epets_tidy_mins_openingdate_01feb2026_CLOSED.pkl'
with open(aus_in, 'rb') as f:
    aus = pickle.load(f, encoding="utf-8")

can_in = '../data/canada/clean/canada_epets_tidy_chron_25aug_CLOSED_25jan2026.pkl'
with open(can_in, 'rb') as f:
    can = pickle.load(f, encoding="utf-8")

In [16]:
aus_keys = list(aus.keys())
can_keys = list(can.keys())

In [17]:
status_field = 'status'
status_filter = ['Responded', 'Presented', 'Closed']
date_field = 'date'
sig_field = 'signatures'
text_fields = ['reason', 'request']
extra_fields = ['minister_clean']
author_field = 'author'
    
# Canada
status_field2 = 'Status'
status_filter2 = ['Government response tabled', 'Not certified', 'Closed upon dissolution', 'Closed', 'Not answered before dissolution', 'Certified', 'Waiting for certification']
date_field2 = 'SignatureOpeningDate_str'
sig_field2 = 'Signatures'
text_fields2 = ['Grievances', 'Prayer']
extra_fields2 = []
author_field2 = 'Author'

## Australia names

In [18]:
# Australia
# create dictionaries with: author first names lowercase, author full name lowercase, author honorific for Aus

def aus_author_dict(epets, keys):
    '''
    Create a dictionary of author information
    '''
    gen_titles = ['mr', 'mstr', 'mister', 'ms', 'miss', 'mrs', 'misss', 'me', 'mt', 'mx']
    male_titles = ['mr', 'mstr', 'mister']
    female_titles = ['ms', 'miss', 'mrs', 'misss']
    unknown_titles = ['me', 'mt', 'mx']

    special_titles = ['sister', 'bishop', 'director', 'dr', 'the', 'professor', 'reverend', 'mp', 'group', 'sir', 'hon', 'captain', 'sr', 'retd', 'lt', 'lord', 'male']
    
    
    a = {'title': None, 'first': None, 'full': None}
    authors = {}
    for k in keys:
        author = {'title': None, 'first': None, 'full': None}
        split_text = epets[k][author_field].lower().replace('.', '').split()
            
        if len(split_text) < 3: # case where there is a first and last name but no title
            author['first'] = split_text[0]
            author['full'] = ' '.join(split_text)
        else:
            if split_text[0] in gen_titles: # cases where there is a gendered title
                author['title'] = split_text[0]
                if len(split_text) == 3: # cases of gendered title + first name + last name
                    author['first'] = split_text[1]
                    author['full'] = ' '.join(split_text[1:])
                else: # length > 3 
                    if split_text[1] not in gen_titles: # cases of gendered title + names
                        author['first'] = split_text[1]
                        author['full'] = ' '.join(split_text[1:])
                    else: # cases of gendered title + gendered title + names (need to remove extra gendered title)
                        author['first'] = split_text[2]
                        author['full'] = ' '.join(split_text[2:])
            elif split_text[0] in special_titles:
                if len(split_text) == 3 or split_text[0] == 'dr': # special title + names
                    author['title'] = split_text[0]
                    author['first'] = split_text[1]
                    author['full'] = ' '.join(split_text[1:])
                else:
                    if split_text[0] == 'lt': # one name to be dealt with
                        author['title'] = ' '.join(split_text[:2])
                        author['first'] = split_text[2]
                        author['full'] = ' '.join(split_text[2:])
                    else: # two names to be dealt with
                        author['title'] = ' '.join(split_text[:3])
                        author['first'] = split_text[3]
                        author['full'] = ' '.join(split_text[3:])
            else:
                author['first'] = split_text[0]
                author['full'] = ' '.join(split_text)

                
        # deal with cases where there is a comma - we want to get rid of everything after the comma
        if ',' in author['full']:
            author['full'] = author['full'].split(',')[0]
        if "(" in author['full']: # removing brackets in names
            author['full']= author['full'].replace('(', '').replace(')', '')
            
        authors[k] = author
    return authors

In [19]:
authors_aus = aus_author_dict(aus, aus_keys)

In [1]:
# Commented out to prevent author names being visible

# for k in authors_aus.keys():
# #     if '?' in authors_aus[k]['full']:
# #         print(authors_aus[k])
#     if len(authors_aus[k]['full'].split()) > 3:
#         print(authors_aus[k])

## Canada names

In [24]:
# Canada
def can_author_dict(epets, keys):
    '''
    Create dictionary of author information
    '''
    a = {'first': None, 'full': None}
    authors = {}
    for k in keys:
        author = {'first': None, 'full': None, 'full_og': None}
        author['full_og'] = epets[k][author_field2].lower()
        split_text = epets[k][author_field2].lower().split()
        split_text = [x for x in split_text if len(x) > 1]
        
        if len(split_text[0].strip('.')) < 2 or split_text[0].strip('.') == 'dr': # cases where first 'name' is just a letter and a '.' e.g 'p.'
            author['first'] = split_text[1]
            author['full'] =' '.join(split_text[1:])
        else:
            author['first'] = split_text[0]
            author['full'] =' '.join(split_text)

        authors[k] = author
        
    return authors

In [25]:
authors_can = can_author_dict(can, can_keys)

In [2]:
# Commented out to prevent author names being visible

# for k in authors_can.keys():
#     if '-' in authors_can[k]['full']:
#         print(authors_can[k])
# #     if len(authors_can[k]['full'].split()) > 3:
# #         print(authors_can[k])

## Save to files - uncomment if running for the first time

In [13]:
# out_aus = './author_analysis/aus_authors_29jan2026.pkl'
# out_can = './author_analysis/can_authors_29jan2026.pkl'

# with open(out_aus, 'wb') as f:
#     pickle.dump(authors_aus, f)
    
# with open(out_can, 'wb') as f:
#     pickle.dump(authors_can, f)